# Isotropic rotational diffusion
The model<sup>1,2</sup> describes a scatterer that stays at a fixed distance $R$ from a centre and moves over the surface of that sphere by rotational diffusion, for example a hydrogen atom in a small molecule that tumbles freely in a liquid. $D_R$ is the rotational diffusion coefficient. The motion is isotropic, so the scattering depends only on the magnitude of $Q$:

$$ S(Q, E) = A_0(Q)\,\delta(E) + \sum_{l=1}^{l_{max}} A_l(Q)\, L(E, \Gamma_l) $$

with

$$ A_0(Q) = j_0^2(QR),\quad A_l(Q) = (2l+1)\, j_l^2(QR),\quad \Gamma_l = l(l+1)\hbar D_R, $$

where $j_l$ are the spherical Bessel functions and $L(E, \Gamma)$ is a Lorentzian with half width at half maximum $\Gamma$. $A_0$ is the Elastic Incoherent Structure Factor (EISF) and $A_l$ the Quasi-Elastic Incoherent Structure Factor (QISF) of order $l$. The widths do not depend on $Q$; only the QISF do.<sup>3</sup>

The sum is infinite. The model keeps the orders up to `l_max`, 6 by default and at most 20, because every order adds Parameters that slow down a fit.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

import easydynamics as edyn

%matplotlib widget

## The model at different Q values

In [ ]:
# Q is in Angstrom^-1, the radius in Angstrom and D_R in 1/ps.
Q = np.linspace(0.3, 2, 7)

rotation_model = edyn.IsotropicRotationalDiffusion(
    scale=1.0,
    radius=2.0,
    rotational_diffusion_coefficient=0.1,
    Q=Q,
)
print(rotation_model)

The radius is fixed by default, since it is normally known from the molecular geometry; free it with `rotation_model.radius.fixed = False` to fit it.

In [ ]:
# Plot the quasi-elastic part (all Lorentzians) at every Q. The delta function
# would be a spike at E = 0; its weight, the EISF, is plotted further down.
energy = np.linspace(-2, 2, 501)
component_collections = rotation_model.get_component_collections()

cmap = plt.cm.jet
nQ = len(component_collections)
plt.figure()
for Q_index, collection in enumerate(component_collections):
    quasielastic = sum(
        component.evaluate(energy)
        for component in collection
        if isinstance(component, edyn.Lorentzian)
    )
    color = cmap(Q_index / (nQ - 1))
    plt.plot(energy, quasielastic, label=fr'Q={Q[Q_index]:.2f} Å^{-1}', color=color)

plt.legend()
plt.xlabel('Energy (meV)')
plt.ylabel('Intensity (arb. units)')
plt.title('Isotropic rotational diffusion, R = 2 Å')
plt.show()

## The widths of the Lorentzians

The width of order $l$ grows as $l(l+1)$, so the higher orders are much broader than the first.

In [ ]:
for order in range(1, rotation_model.l_max + 1):
    width = rotation_model.calculate_width(order=order)[0]
    print(f'order {order}: HWHM = {width:.4f} meV')

## EISF and QISF as a function of Q

At small $QR$ almost all intensity is elastic. As $QR$ grows, the weight moves into the Lorentzians, one order after the other: the QISF of order $l$ is small until $QR$ approaches $l$. The EISF and the QISF of all orders sum to one.

In [ ]:
Q_fine = np.linspace(0.01, 2, 200)

plt.figure()
plt.plot(Q_fine, rotation_model.calculate_EISF(Q_fine), color='k', label='EISF')
for order in range(1, 5):
    plt.plot(
        Q_fine,
        rotation_model.calculate_QISF_order(Q_fine, order=order),
        label=f'QISF of order {order}',
    )
plt.plot(
    Q_fine,
    rotation_model.calculate_EISF(Q_fine) + rotation_model.calculate_QISF(Q_fine),
    color='k',
    linestyle='dashed',
    label='EISF + QISF',
)
plt.xlabel('Q (Å$^{-1}$)')
plt.ylabel('Weight')
plt.title('EISF and QISF for R = 2 Å')
plt.legend()
plt.show()

## Truncation

Leaving out the orders above `l_max` loses intensity at large $QR$. A warning is raised when more than $10^{-3}$ of the weight is lost at any of the model's $Q$ values, and `calculate_missing_weight` returns the weight that is left out. With the default `l_max` of 6 this starts at $QR \approx 4.3$, and with the largest, 20, at $QR \approx 17$. A model without $Q$ values builds no components, so it is quick to create for this check.

In [ ]:
QR = np.linspace(0.01, 20, 400)

plt.figure()
for l_max in [2, 6, 20]:
    model = edyn.IsotropicRotationalDiffusion(radius=1.0, l_max=l_max)
    plt.semilogy(
        QR,
        np.maximum(model.calculate_missing_weight(QR), 1e-16),
        label=f'l_max = {l_max}',
    )
plt.axhline(1e-3, color='k', linestyle='dashed', label='warning threshold')
plt.xlabel('QR')
plt.ylabel('Missing weight')
plt.title('Weight left out by the truncated sum')
plt.legend()
plt.show()

Too small an `l_max` for the $Q$ range raises the warning:

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    short_model = edyn.IsotropicRotationalDiffusion(radius=2.0, l_max=2, Q=np.linspace(0.3, 2, 4))
for warning in caught:
    print(warning.message)

print('Missing weight per Q:', np.round(short_model.calculate_missing_weight(), 4))

More orders remove the warning but also make every update of the model slower. The spherical Bessel functions are faster with the optional numba dependency, `pip install easydynamics[fast]`; without it SciPy gives the same values.

## References

1. V. F. Sears, *Can. J. Phys.* **44**, 1299 (1966), [doi:10.1139/p66-109](https://doi.org/10.1139/p66-109).

2. V. F. Sears, *Can. J. Phys.* **45**, 237 (1967), [doi:10.1139/p67-025](https://doi.org/10.1139/p67-025).

3. M. Bée, *Quasielastic Neutron Scattering*, Adam Hilger, Bristol (1988).